# QLoRA fine-tuning for financial-news sentiment: does it beat the simple baselines?

A 1.5B-parameter model is fine-tuned with 4-bit QLoRA on a labelled set of financial-news headlines
(Bearish / Bullish / Neutral) and compared **on the same held-out test set** against:

0. the **majority class** (the test set is about two-thirds Neutral, so this is the floor),
1. a hand-written keyword **rules** baseline,
2. **TF-IDF + logistic regression** (the strong classical baseline),
3. the **same base model, zero-shot**,
4. the **same base model, few-shot** (8 demonstrations).

Everything is scored the same way (next-token probabilities over the three label words), with bootstrap
confidence intervals and paired exact McNemar tests (Holm-adjusted). Nothing is tuned on the test set.

**How to run:** Runtime → Change runtime type → **T4 GPU**, then *Run all*. Expected time: about 35-50 minutes
(training is about 15-25 of those). The last cell writes `fxnews_results.zip`; download it and keep it, the
repository's README is filled from `results.json` inside it.

## 1. Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.46,<5", "peft>=0.13", "bitsandbytes>=0.44", "accelerate>=1.0",
                "datasets>=3.0", "scikit-learn", "scipy", "matplotlib", "pandas"], check=True)
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.__version__, torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. The library (identical to the unit-tested source in `src/fxnews`)

In [ ]:
import os
os.makedirs('fxnews', exist_ok=True)

In [ ]:
%%writefile fxnews/__init__.py

In [ ]:
%%writefile fxnews/data.py
"""Data for the sentiment task and the split protocol.

Task: classify a short financial-news headline as Bearish, Bullish or Neutral for the asset it mentions.
Dataset: `zeroshot/twitter-financial-news-sentiment` on the Hugging Face Hub (about 11.9k labelled tweets
from financial news accounts; labels 0 Bearish, 1 Bullish, 2 Neutral). The Hub page states the licence; check it
before any use beyond research, and it is downloaded by the notebook, never stored in this repository.

Protocol (fixed before any model is trained):
  * the dataset's own `validation` split is the TEST set, touched once at the end;
  * `train` is split 90/10 (stratified, seeded) into fit and DEV, and DEV is the only place anything is tuned;
  * any text that appears in both train and test (exact match after normalisation) is removed from train, and
    the number removed is reported, because duplicated headlines inflate every model's score.
"""

from __future__ import annotations

import re
from dataclasses import dataclass

import numpy as np

LABELS = ["Bearish", "Bullish", "Neutral"]
HUB_NAME = "zeroshot/twitter-financial-news-sentiment"
SEED = 42


def normalise(text: str) -> str:
    text = re.sub(r"https?://\S+", " ", text.lower())
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9$%.' ]+", " ", text)).strip()


@dataclass
class Splits:
    fit_x: list[str]
    fit_y: np.ndarray
    dev_x: list[str]
    dev_y: np.ndarray
    test_x: list[str]
    test_y: np.ndarray
    removed_duplicates: int


def make_splits(train_x: list[str], train_y: list[int], test_x: list[str], test_y: list[int], dev_frac: float = 0.1, seed: int = SEED) -> Splits:
    ty = np.asarray(train_y)
    test_norm = {normalise(t) for t in test_x}
    keep = np.array([normalise(t) not in test_norm for t in train_x])
    removed = int((~keep).sum())
    tx = [t for t, k in zip(train_x, keep, strict=True) if k]
    ty = ty[keep]
    rng = np.random.default_rng(seed)
    dev_idx = []
    for c in np.unique(ty):  # stratified: every class keeps the same share in DEV
        idx = np.flatnonzero(ty == c)
        rng.shuffle(idx)
        dev_idx += idx[: max(1, int(round(len(idx) * dev_frac)))].tolist()
    mask = np.zeros(len(tx), dtype=bool)
    mask[dev_idx] = True
    return Splits([t for t, m in zip(tx, mask, strict=True) if not m], ty[~mask], [t for t, m in zip(tx, mask, strict=True) if m], ty[mask],
                  list(test_x), np.asarray(test_y), removed)


def load_hub():
    """Returns (train_x, train_y, test_x, test_y). Needs the `datasets` package and network access."""
    from datasets import load_dataset

    ds = load_dataset(HUB_NAME)
    return ds["train"]["text"], ds["train"]["label"], ds["validation"]["text"], ds["validation"]["label"]

In [ ]:
%%writefile fxnews/metrics.py
"""Metrics and the tests used to compare models. Everything here is plain NumPy/SciPy so it is unit-tested on a laptop."""

from __future__ import annotations

import numpy as np
from scipy import stats


def accuracy(y: np.ndarray, p: np.ndarray) -> float:
    return float(np.mean(np.asarray(y) == np.asarray(p)))


def macro_f1(y: np.ndarray, p: np.ndarray, n_classes: int = 3) -> float:
    y, p = np.asarray(y), np.asarray(p)
    f1s = []
    for c in range(n_classes):
        tp = np.sum((y == c) & (p == c))
        fp = np.sum((y != c) & (p == c))
        fn = np.sum((y == c) & (p != c))
        denom = 2 * tp + fp + fn
        f1s.append(2 * tp / denom if denom else 0.0)
    return float(np.mean(f1s))


def confusion(y: np.ndarray, p: np.ndarray, n_classes: int = 3) -> np.ndarray:
    m = np.zeros((n_classes, n_classes), dtype=int)
    for a, b in zip(np.asarray(y), np.asarray(p), strict=True):
        m[a, b] += 1
    return m


def bootstrap_ci(y: np.ndarray, p: np.ndarray, fn, reps: int = 2000, seed: int = 0) -> tuple[float, float, float]:
    """Percentile bootstrap over test examples for any metric fn(y, p)."""
    y, p = np.asarray(y), np.asarray(p)
    rng = np.random.default_rng(seed)
    vals = np.empty(reps)
    for i in range(reps):
        idx = rng.integers(0, len(y), len(y))
        vals[i] = fn(y[idx], p[idx])
    return float(fn(y, p)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def mcnemar(y: np.ndarray, a: np.ndarray, b: np.ndarray) -> tuple[int, int, float]:
    """Exact McNemar test: do models a and b differ in accuracy on the same examples? Returns (a-only correct, b-only correct, p)."""
    ok_a, ok_b = np.asarray(a) == np.asarray(y), np.asarray(b) == np.asarray(y)
    n10, n01 = int(np.sum(ok_a & ~ok_b)), int(np.sum(~ok_a & ok_b))
    if n10 + n01 == 0:
        return n10, n01, 1.0
    return n10, n01, float(min(1.0, stats.binomtest(n10, n10 + n01, 0.5).pvalue))


def holm(pvalues: list[float]) -> list[float]:
    m = len(pvalues)
    order = np.argsort(pvalues)
    adj, running = np.empty(m), 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (m - rank) * pvalues[i]))
        adj[i] = running
    return adj.tolist()

In [ ]:
%%writefile fxnews/baselines.py
"""The non-LLM baselines every fine-tuned model has to be compared with.

* Rules: a hand-written keyword lexicon. It is the "could a few lines of code do this" baseline. Written by the
  repo author before looking at any model output; the word lists are short on purpose.
* TF-IDF + logistic regression: the strong classical baseline. The regularisation strength is chosen on DEV only.
"""

from __future__ import annotations

import re

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

BEARISH = {"falls", "fall", "drops", "drop", "slump", "slumps", "plunge", "plunges", "tumble", "tumbles", "sinks", "cuts", "cut", "misses", "miss",
           "downgrade", "downgrades", "downgraded", "lower", "weak", "warns", "warning", "loss", "losses", "lawsuit", "probe", "recall", "bankruptcy",
           "layoffs", "decline", "declines", "sell", "bearish", "short"}
BULLISH = {"rises", "rise", "jumps", "jump", "surges", "surge", "soars", "soar", "rallies", "rally", "gains", "gain", "beats", "beat", "upgrade",
           "upgrades", "upgraded", "higher", "strong", "record", "raises", "raise", "profit", "wins", "approval", "approved", "buy", "bullish", "growth"}


def rule_predict(text: str) -> int:
    """0 Bearish, 1 Bullish, 2 Neutral: whichever lexicon has more hits, Neutral on a tie or no hits."""
    words = set(re.findall(r"[a-z]+", text.lower()))
    bear, bull = len(words & BEARISH), len(words & BULLISH)
    return 0 if bear > bull else 1 if bull > bear else 2


def majority(fit_y, n: int) -> np.ndarray:
    """Always predict the most frequent class in the fit split. The floor that any model must clear to be saying anything."""
    return np.full(n, int(np.bincount(np.asarray(fit_y)).argmax()))


def rules(texts: list[str]) -> np.ndarray:
    return np.array([rule_predict(t) for t in texts])


def fit_tfidf(fit_x: list[str], fit_y, dev_x: list[str], dev_y, grid=(0.3, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0)):
    """Choose C on DEV (macro-F1), then refit on fit+dev. Returns (model, chosen C, DEV score per C)."""
    from fxnews.metrics import macro_f1

    scores = {}
    for c in grid:
        m = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), LogisticRegression(C=c, max_iter=2000))
        m.fit(fit_x, fit_y)
        scores[c] = macro_f1(dev_y, m.predict(dev_x))
    best = max(scores, key=scores.get)
    final = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), LogisticRegression(C=best, max_iter=2000))
    final.fit(list(fit_x) + list(dev_x), np.concatenate([np.asarray(fit_y), np.asarray(dev_y)]))
    return final, best, scores

In [ ]:
%%writefile fxnews/summary.py
"""Turn predictions into the results table: point estimates with bootstrap intervals, per-class F1, paired tests."""

from __future__ import annotations

import numpy as np

from fxnews import metrics
from fxnews.data import LABELS


def per_class_f1(y: np.ndarray, p: np.ndarray) -> dict[str, float]:
    out = {}
    for c, name in enumerate(LABELS):
        tp, fp, fn = np.sum((y == c) & (p == c)), np.sum((y != c) & (p == c)), np.sum((y == c) & (p != c))
        out[name] = float(2 * tp / (2 * tp + fp + fn)) if (2 * tp + fp + fn) else 0.0
    return out


def summarise(y: np.ndarray, pred: np.ndarray, ms_per_example: float | None = None, reps: int = 2000) -> dict:
    y, pred = np.asarray(y), np.asarray(pred)
    return {
        "accuracy": metrics.bootstrap_ci(y, pred, metrics.accuracy, reps),
        "macro_f1": metrics.bootstrap_ci(y, pred, metrics.macro_f1, reps),
        "per_class_f1": per_class_f1(y, pred),
        "confusion": metrics.confusion(y, pred).tolist(),
        "ms_per_example": ms_per_example,
    }


def compare(y: np.ndarray, preds: dict[str, np.ndarray], reference: str) -> dict[str, dict]:
    """Paired exact McNemar of every model against `reference`, Holm-adjusted across the comparisons."""
    names = [n for n in preds if n != reference]
    raw = [metrics.mcnemar(y, preds[n], preds[reference]) for n in names]  # a = model, b = reference
    adj = metrics.holm([r[2] for r in raw])
    return {n: {"model_only_correct": r[0], "reference_only_correct": r[1], "p": r[2], "p_holm": a} for n, r, a in zip(names, raw, adj, strict=True)}


def markdown_table(results: dict[str, dict], extras: dict[str, dict[str, str]] | None = None) -> str:
    """`results`: name -> summarise() output. `extras`: name -> {column: text} (e.g. training time, GPU memory)."""
    extra_cols = sorted({k for v in (extras or {}).values() for k in v})
    head = ["model", "accuracy (95% CI)", "macro-F1 (95% CI)", "ms / example", *extra_cols]
    lines = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
    for name, r in results.items():
        a, f = r["accuracy"], r["macro_f1"]
        ms = "" if r["ms_per_example"] is None else f"{r['ms_per_example']:.1f}"
        row = [name, f"{a[0]:.3f} ({a[1]:.3f}-{a[2]:.3f})", f"{f[0]:.3f} ({f[1]:.3f}-{f[2]:.3f})", ms, *[(extras or {}).get(name, {}).get(c, "") for c in extra_cols]]
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines)

In [ ]:
%%writefile fxnews/llm.py
"""Prompting, label scoring and QLoRA training for a small causal LM.

All three LLM variants (zero-shot, few-shot, fine-tuned) are scored the same way: the model's next-token
probabilities over the three label words, read from ONE forward pass. This avoids free-text parsing (no
"the answer seems to be bullish-ish" ambiguity) and makes the comparison between prompting and fine-tuning
a comparison of the models, not of how forgiving the output parser is.

The torch/transformers/peft imports are inside functions so the pure parts (prompts, example construction,
collation) can be unit-tested on a machine without a GPU or those libraries.
"""

from __future__ import annotations

from collections.abc import Sequence

import numpy as np

from fxnews.data import LABELS

INSTRUCTION = "Classify the sentiment of this financial news headline for the asset it mentions as Bearish, Bullish or Neutral."
MAX_LEN = 128


def build_prompt(text: str, shots: Sequence[tuple[str, int]] = ()) -> str:
    """Plain-text prompt (no chat template, so it behaves the same across model families). The answer follows 'Sentiment:'."""
    parts = [INSTRUCTION, ""]
    for t, y in shots:
        parts += [f"Headline: {t}", f"Sentiment: {LABELS[y]}", ""]
    parts += [f"Headline: {text}", "Sentiment:"]
    return "\n".join(parts)


def choose_shots(texts: Sequence[str], labels: np.ndarray, per_class: Sequence[int] = (3, 3, 2), seed: int = 0) -> list[tuple[str, int]]:
    """A fixed, class-balanced set of demonstrations drawn from the FIT split (never from DEV or test)."""
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    shots = []
    for c, k in enumerate(per_class):
        idx = np.flatnonzero(labels == c)
        shots += [(texts[i], c) for i in rng.choice(idx, k, replace=False)]
    order = rng.permutation(len(shots))  # interleave classes so the last demonstration is not always the same label
    return [shots[i] for i in order]


def make_sft_example(tokenizer, text: str, label: int, max_len: int = MAX_LEN) -> dict[str, list[int]]:
    """Prompt tokens are masked out of the loss (-100); only the label word and the end-of-sequence token are learned."""
    prompt_ids = tokenizer(build_prompt(text), add_special_tokens=False)["input_ids"][: max_len - 8]
    target_ids = tokenizer(" " + LABELS[label], add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
    return {"input_ids": prompt_ids + target_ids, "labels": [-100] * len(prompt_ids) + target_ids}


def collate(batch: list[dict[str, list[int]]], pad_id: int) -> dict[str, list[list[int]]]:
    """Right-pad to the longest sequence in the batch."""
    n = max(len(b["input_ids"]) for b in batch)
    return {
        "input_ids": [b["input_ids"] + [pad_id] * (n - len(b["input_ids"])) for b in batch],
        "attention_mask": [[1] * len(b["input_ids"]) + [0] * (n - len(b["input_ids"])) for b in batch],
        "labels": [b["labels"] + [-100] * (n - len(b["labels"])) for b in batch],
    }


def label_token_ids(tokenizer) -> list[int]:
    """First token of each ' <Label>'. They must differ, otherwise one forward pass cannot tell the labels apart."""
    ids = [tokenizer(" " + lab, add_special_tokens=False)["input_ids"][0] for lab in LABELS]
    if len(set(ids)) != len(ids):
        raise ValueError(f"label words share a first token ({ids}); pick different label words for this tokenizer")
    return ids


def load_model(name: str, four_bit: bool = True):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    tok = AutoTokenizer.from_pretrained(name)
    tok.padding_side = "left"  # so the last position of every row is the position that predicts the answer
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kwargs = {"device_map": {"": 0}, "torch_dtype": torch.float16}
    if four_bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(name, **kwargs), tok


def score(model, tok, texts: Sequence[str], shots: Sequence[tuple[str, int]] = (), batch_size: int = 32) -> tuple[np.ndarray, np.ndarray, float]:
    """Returns (probabilities [n, 3], predictions [n], milliseconds per example)."""
    import time

    import torch

    ids = torch.tensor(label_token_ids(tok), device=model.device)
    model.eval()
    probs = []
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            enc = tok([build_prompt(t, shots) for t in texts[i : i + batch_size]], return_tensors="pt", padding=True, truncation=True,
                      max_length=1024, add_special_tokens=False).to(model.device)
            logits = model(**enc).logits[:, -1, :]
            probs.append(torch.softmax(logits[:, ids].float(), dim=-1).cpu().numpy())
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) * 1000 / len(texts)
    p = np.concatenate(probs)
    return p, p.argmax(1), ms


def add_lora(model, r: int = 16, alpha: int = 32, dropout: float = 0.05):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    model = prepare_model_for_kbit_training(model)
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, bias="none", task_type="CAUSAL_LM",
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    return get_peft_model(model, cfg)


def train_lora(model, tok, fit_x, fit_y, dev_x, dev_y, out_dir: str, epochs: int = 2, lr: float = 2e-4, batch_size: int = 16, grad_accum: int = 2,
               seed: int = 42):
    """Plain `Trainer` (no TRL), so the training loop does not depend on a fast-moving wrapper API. Returns the trainer."""
    import torch
    from transformers import Trainer, TrainingArguments

    class DS(torch.utils.data.Dataset):
        def __init__(self, xs, ys):
            self.items = [make_sft_example(tok, x, int(y)) for x, y in zip(xs, ys, strict=True)]

        def __len__(self):
            return len(self.items)

        def __getitem__(self, i):
            return self.items[i]

    def collator(batch):
        return {k: torch.tensor(v) for k, v in collate(batch, tok.pad_token_id).items()}

    args = TrainingArguments(
        output_dir=out_dir, num_train_epochs=epochs, per_device_train_batch_size=batch_size, per_device_eval_batch_size=batch_size * 2,
        gradient_accumulation_steps=grad_accum, learning_rate=lr, lr_scheduler_type="cosine", warmup_ratio=0.05, weight_decay=0.0, fp16=True,
        optim="paged_adamw_8bit", gradient_checkpointing=True, logging_steps=25, eval_strategy="steps", eval_steps=200, save_strategy="no",
        report_to="none", seed=seed, remove_unused_columns=False,
    )
    model.config.use_cache = False
    trainer = Trainer(model=model, args=args, train_dataset=DS(fit_x, fit_y), eval_dataset=DS(dev_x, dev_y), data_collator=collator)
    trainer.train()
    model.config.use_cache = True
    return trainer

## 3. Configuration
All choices are made **here, once, before looking at any result**.

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # Apache-2.0. A 3B model also fits on a T4: "Qwen/Qwen2.5-3B-Instruct"
EPOCHS, LR, LORA_R, LORA_ALPHA = 2, 2e-4, 16, 32
SEED = 42
import random, numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 4. Data and split protocol
The dataset's own `validation` split is the **test set** and is used once, at the end. `train` is split 90/10 into
*fit* and *dev*; only dev is used for choices. Headlines that appear in both train and test are removed from train
(the number is reported).

In [ ]:
import numpy as np, pandas as pd
from fxnews import data
tx, ty, vx, vy = data.load_hub()
S = data.make_splits(tx, ty, vx, vy)
print(f"fit {len(S.fit_x)} | dev {len(S.dev_x)} | test {len(S.test_x)} | duplicates removed from train: {S.removed_duplicates}")
print(pd.DataFrame({"fit": np.bincount(S.fit_y), "dev": np.bincount(S.dev_y), "test": np.bincount(S.test_y)}, index=data.LABELS))

## 5. Baselines: rules and TF-IDF + logistic regression (CPU)

In [ ]:
import time
from fxnews import baselines, summary
results, preds, extras, probs = {}, {}, {}, {}

preds["majority class"] = baselines.majority(S.fit_y, len(S.test_x))
results["majority class"] = summary.summarise(S.test_y, preds["majority class"], 0.0)
preds["rules"] = baselines.rules(S.test_x)
results["rules"] = summary.summarise(S.test_y, preds["rules"], 0.0)

tfidf, C, dev_scores = baselines.fit_tfidf(S.fit_x, S.fit_y, S.dev_x, S.dev_y)
t0 = time.perf_counter(); preds["tfidf+logreg"] = tfidf.predict(S.test_x)
results["tfidf+logreg"] = summary.summarise(S.test_y, preds["tfidf+logreg"], (time.perf_counter() - t0) * 1000 / len(S.test_x))
print("TF-IDF C chosen on dev:", C, {k: round(v, 3) for k, v in dev_scores.items()})
for k in ("majority class", "rules", "tfidf+logreg"):
    print(k, results[k]["accuracy"][0], results[k]["macro_f1"][0])

## 6. The base model: zero-shot and few-shot
The base model is loaded once in 4-bit. Both prompting variants read the label from the next-token probabilities.
The 8 demonstrations come from the *fit* split (class-balanced, fixed seed).

In [ ]:
from fxnews import llm
model, tok = llm.load_model(MODEL, four_bit=True)
print("label first-token ids:", llm.label_token_ids(tok))

p, pred, ms = llm.score(model, tok, S.test_x);               preds["zero-shot"] = pred; probs["zero-shot"] = p
results["zero-shot"] = summary.summarise(S.test_y, pred, ms)
SHOTS = llm.choose_shots(S.fit_x, S.fit_y, (3, 3, 2), seed=SEED)
p, pred, ms = llm.score(model, tok, S.test_x, shots=SHOTS);  preds["few-shot (8)"] = pred; probs["few-shot (8)"] = p
results["few-shot (8)"] = summary.summarise(S.test_y, pred, ms)
for k in ("zero-shot", "few-shot (8)"):
    print(k, round(results[k]["accuracy"][0], 3), round(results[k]["macro_f1"][0], 3), f"{results[k]['ms_per_example']:.1f} ms/ex")

## 7. QLoRA fine-tuning

In [ ]:
import time, torch
model = llm.add_lora(model, r=LORA_R, alpha=LORA_ALPHA)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad); total = sum(p.numel() for p in model.parameters())
print(f"trainable parameters: {trainable/1e6:.1f}M of {total/1e6:.0f}M ({100*trainable/total:.2f}%)")
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer = llm.train_lora(model, tok, S.fit_x, S.fit_y, S.dev_x, S.dev_y, "qlora_out", epochs=EPOCHS, lr=LR, seed=SEED)
train_minutes = (time.time() - t0) / 60
peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"training: {train_minutes:.1f} min, peak GPU memory {peak_gb:.1f} GB")

In [ ]:
import matplotlib.pyplot as plt
log = trainer.state.log_history
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot([l["step"] for l in log if "loss" in l], [l["loss"] for l in log if "loss" in l], label="train loss")
ev = [(l["step"], l["eval_loss"]) for l in log if "eval_loss" in l]
if ev: ax.plot(*zip(*ev), marker="o", label="dev loss")
ax.set_xlabel("step"); ax.legend(); fig.tight_layout(); fig.savefig("training_curve.png", dpi=130); plt.show()

## 8. Final evaluation on the test set (used once)

In [ ]:
p, pred, ms = llm.score(model, tok, S.test_x);  preds["QLoRA fine-tuned"] = pred; probs["QLoRA fine-tuned"] = p
results["QLoRA fine-tuned"] = summary.summarise(S.test_y, pred, ms)
extras = {"QLoRA fine-tuned": {"train min": f"{train_minutes:.0f}", "peak GPU GB": f"{peak_gb:.1f}", "trainable params": f"{trainable/1e6:.1f}M"}}
order = ["majority class", "rules", "tfidf+logreg", "zero-shot", "few-shot (8)", "QLoRA fine-tuned"]
table = summary.markdown_table({k: results[k] for k in order}, extras)
print(table)

In [ ]:
cmp = summary.compare(S.test_y, {k: preds[k] for k in order}, reference="tfidf+logreg")
print("Paired exact McNemar versus TF-IDF + logistic regression (Holm-adjusted):")
for k, v in cmp.items():
    print(f"  {k:18s} model-only-correct {v['model_only_correct']:4d} | tfidf-only-correct {v['reference_only_correct']:4d} | p {v['p']:.4f} | p_holm {v['p_holm']:.4f}")
cmp_ft = summary.compare(S.test_y, {k: preds[k] for k in ("few-shot (8)", "QLoRA fine-tuned")}, reference="few-shot (8)")
print("Fine-tuned vs few-shot:", cmp_ft)
for k in ("tfidf+logreg", "few-shot (8)", "QLoRA fine-tuned"):
    print(k, "per-class F1:", {a: round(b, 3) for a, b in results[k]["per_class_f1"].items()})

## 9. Error analysis: what does the fine-tuned model still get wrong?

In [ ]:
import numpy as np
wrong = np.flatnonzero(preds["QLoRA fine-tuned"] != S.test_y)
rng = np.random.default_rng(0)
for i in rng.choice(wrong, min(15, len(wrong)), replace=False):
    pr = probs["QLoRA fine-tuned"][i]
    print(f"gold {data.LABELS[S.test_y[i]]:8s} | pred {data.LABELS[preds['QLoRA fine-tuned'][i]]:8s} ({pr.max():.2f}) | {S.test_x[i][:140]}")
print(f"\n{len(wrong)} errors of {len(S.test_y)}")

## 10. Optional ablation (off by default; about 20 minutes each)
Set `RUN_ABLATION = True` to compare LoRA rank 4 against 16 on **dev** (never test). It reloads the base model, so
run it last.

In [ ]:
RUN_ABLATION = False
if RUN_ABLATION:
    import gc
    ablation = {}
    for r in (4, 16):
        del model; gc.collect(); torch.cuda.empty_cache()
        m, t = llm.load_model(MODEL, four_bit=True)
        m = llm.add_lora(m, r=r, alpha=2 * r)
        llm.train_lora(m, t, S.fit_x, S.fit_y, S.dev_x, S.dev_y, f"abl_r{r}", epochs=EPOCHS, lr=LR, seed=SEED)
        _, pd_, _ = llm.score(m, t, S.dev_x)
        from fxnews import metrics
        ablation[f"r={r}"] = metrics.macro_f1(S.dev_y, pd_)
        model = m
    print("dev macro-F1:", ablation)

## 11. Save everything

In [ ]:
import json, shutil, platform
out = {
    "model": MODEL, "seed": SEED, "epochs": EPOCHS, "lr": LR, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
    "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
    "n": {"fit": len(S.fit_x), "dev": len(S.dev_x), "test": len(S.test_x), "duplicates_removed": S.removed_duplicates},
    "results": results, "comparisons_vs_tfidf": cmp, "table_markdown": table,
    "predictions": {k: [int(x) for x in v] for k, v in preds.items()}, "test_labels": [int(x) for x in S.test_y],
}
json.dump(out, open("results.json", "w"), indent=2)
model.save_pretrained("qlora_adapter")           # the adapter only (tens of MB), not the base model
shutil.make_archive("fxnews_results", "zip", ".", "qlora_adapter")
import zipfile
with zipfile.ZipFile("fxnews_results.zip", "a") as z:
    for f in ("results.json", "training_curve.png"): z.write(f)
print("wrote fxnews_results.zip  ->  download it from the Files panel")
try:
    from google.colab import files; files.download("fxnews_results.zip")
except Exception:
    pass